In [8]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
for mv in client.search_model_versions("name='Mejor modelo'"):
    print(f"Version: {mv.version}, Stage: {mv.current_stage}, Run ID: {mv.run_id}")


Version: 2, Stage: None, Run ID: 80cc7c7c7db14aa0934e0762379402df
Version: 1, Stage: None, Run ID: 80cc7c7c7db14aa0934e0762379402df


In [9]:
import mlflow
from mlflow.tracking import MlflowClient
from feature_engineer import add_features
from etl import Dataloader
import pandas as pd

def get_model():
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    client = MlflowClient()

    model_name = "Mejor modelo"

    try:
        # 🔹 Intentar primero con alias (nuevo enfoque recomendado)
        model_uri = f"models:/{model_name}@production"
        model = mlflow.sklearn.load_model(model_uri)
        print(f"✅ Modelo cargado con alias: {model_uri}")
        return model
    except Exception as e_alias:
        print(f"⚠️ No se encontró alias 'production': {e_alias}")
        print("🔄 Intentando cargar con stage 'Production'...")

        try:
            # 🔹 Intentar con stage clásico (modo legacy)
            model_uri = f"models:/{model_name}/Production"
            model = mlflow.sklearn.load_model(model_uri)
            print(f"✅ Modelo cargado con stage: {model_uri}")
            return model
        except Exception as e_stage:
            raise RuntimeError(
                f"❌ No se encontró el modelo '{model_name}' ni en alias 'production' "
                f"ni en stage 'Production'. Error original: {e_stage}"
            )

def get_etl_data():
    dataloader = Dataloader(path_to_save="data/pred_batch.csv", n_samples=100000)
    dataset_path = dataloader.download_data()
    df = dataloader.load_data()
    return df

def get_features(df):
    feature_engineer = add_features(df)
    df = feature_engineer.preprocess_data()
    return df

def predict(model, df):
    return model.predict_proba(df)

def save_prediction(df, prediction):
    df["prediction"] = prediction[:, 1]
    df["prediction"] = df["prediction"].astype(int)
    df["prediction"] = df["prediction"].map({0: "No", 1: "Si"})
    df.to_csv('predictions/predictions.csv', index=False)
    return df


In [10]:
from etl import Dataloader

def get_etl_data():
    dataloader = Dataloader(path_to_save="C:/Users/Valentina Molina/Documents/Repositorios/Proyecto_Final_MLOps/data/PS_20174392719_1491204439457_log.csv")
    df = dataloader.load_data()
    dataloader.drop_name_columns()
    return df


In [11]:
import os

print("Directorio actual:", os.getcwd())
print("Archivos en carpeta predictions:", os.listdir("predictions") if os.path.exists("predictions") else "Carpeta no existe")


Directorio actual: c:\Users\Valentina Molina\Documents\repositorios\Proyecto_Final_MLOps\src\app\pred_batch
Archivos en carpeta predictions: []


In [12]:
model = get_model()

✅ Modelo cargado con alias: models:/Mejor modelo@production


In [13]:
model

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
df = get_etl_data()

In [17]:
from feature_engineer import add_features
df = add_features(df)
df.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,diff_old_new_orig,diff_old_new_dest,amount_to_orig_balance,amount_to_dest_balance
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,9839.64,0.0,0.057834,9839.640000
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,1864.28,0.0,0.087731,1864.280000
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,181.00,0.0,0.994505,181.000000
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,181.00,21182.0,0.994505,0.008545
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,11668.14,0.0,0.280788,11668.140000


In [18]:
prediction = predict(model, df)

In [19]:
prediction

array([[1.        , 0.        ],
       [1.        , 0.        ],
       [0.14689381, 0.85310619],
       ...,
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ]], shape=(100000, 2))

In [20]:
prediction_array = prediction[:, 1]

In [21]:
save_prediction(df, prediction)

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,diff_old_new_orig,diff_old_new_dest,amount_to_orig_balance,amount_to_dest_balance,prediction
0,1,PAYMENT,9839.64,170136.0,160296.36,0.00,0.00,0,9839.64,0.00,0.057834,9839.640000,No
1,1,PAYMENT,1864.28,21249.0,19384.72,0.00,0.00,0,1864.28,0.00,0.087731,1864.280000,No
2,1,TRANSFER,181.00,181.0,0.00,0.00,0.00,1,181.00,0.00,0.994505,181.000000,No
3,1,CASH_OUT,181.00,181.0,0.00,21182.00,0.00,1,181.00,21182.00,0.994505,0.008545,No
4,1,PAYMENT,11668.14,41554.0,29885.86,0.00,0.00,0,11668.14,0.00,0.280788,11668.140000,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,10,PAYMENT,4020.66,159929.0,155908.34,0.00,0.00,0,4020.66,0.00,0.025140,4020.660000,No
99996,10,PAYMENT,18345.49,6206.0,0.00,0.00,0.00,0,6206.00,0.00,2.955613,18345.490000,No
99997,10,CASH_IN,183774.91,39173.0,222947.91,54925.05,0.00,0,-183774.91,54925.05,4.691247,3.345861,No
99998,10,CASH_OUT,82237.17,6031.0,0.00,592635.66,799140.46,0,6031.00,-206504.80,13.633483,0.138765,No
